# 04｜線性模型與最佳化

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/04_線性模型與最佳化.md`，目前 52 頁。每個程式格首行標出「程式 04-xx」及頁次；本檔完整對照見 `programs/SLIDE_NOTEBOOK_MAP_02_04.md`。

各格會標示實驗資料、評估方式與適用範圍；課堂手算例使用題目中的固定輸入直接驗算。

| 投影片頁次 | 程式代碼 | 學習內容 |
|---|---|---|
| 5、7、8、9、15、16、17、18 | 04-03 | 簡單線性迴歸：一個特徵預測一個數值；多元線性迴歸：特徵加權與矩陣表示；訓練目標：讓所有殘差的平方平均最小；教材例子：帶有雜訊的直線資料；從「梯度為零」得到正規方程；估計直線與資料生成直線不是同一條；SVD 與偽反矩陣：處理共線與非方陣；程式：避免自己計算反矩陣 |
| 9、16 | 04-04 | 教材例子：帶有雜訊的直線資料；估計直線與資料生成直線不是同一條 |
| 17、50 | 04-05 | SVD 與偽反矩陣：處理共線與非方陣；Notebook 導讀：解析梯度與有限差分 |
| 8、20 | 04-06 | 訓練目標：讓所有殘差的平方平均最小；手算：三個點能決定哪一條線？ |
| 10、11、12、48、50 | 04-07 | 導數與偏導數：參數改變時，損失怎麼變？；平方損失的偏微分：截距與斜率；簡單迴歸的閉式解：偏導數設為零；延伸閱讀：閉式解中的置中求和；Notebook 導讀：解析梯度與有限差分 |
| 7、13、14 | 04-08 | 多元線性迴歸：特徵加權與矩陣表示；多元迴歸的偏導數：每個權重各算一次；多元迴歸的平方損失與梯度 |
| 3、22、23、24、25 | 04-09 | 程式實驗與實際輸出；梯度下降：根據目前斜率移動參數；梯度更新：同一組更新前參數，同步調整；學習率太小：方向對，卻走得很慢；學習率太大：可能越走越遠 |
| 22、24、25、26 | 04-10 | 梯度下降：根據目前斜率移動參數；學習率太小：方向對，卻走得很慢；學習率太大：可能越走越遠；一般損失曲面可能有局部低點與平臺 |
| 27 | 04-11 | 特徵尺度不同，會把等高線拉長 |
| 30 | 04-12 | 同一資料，不同學習率的更新軌跡 |
| 23、28、29 | 04-13 | 梯度更新：同一組更新前參數，同步調整；手算一次更新，檢查損失是否下降；手算解答：從殘差到新的預測 |
| 23、49 | 04-14 | 梯度更新：同一組更新前參數，同步調整；延伸閱讀：負梯度方向的一階近似 |
| 31、32、34、36、37 | 04-15 | 隨機梯度：以一筆資料估計下降方向；SGD 的學習率可逐步衰減；Batch、SGD 與 Mini-batch 的路徑比較；表 4-1：資料規模與可用訓練方式；表 4-1：超參數、縮放與實作 |
| 31、33、34、51 | 04-16 | 隨機梯度：以一筆資料估計下降方向；SGD 的前幾步：單筆改善不等於整體改善；Batch、SGD 與 Mini-batch 的路徑比較；Notebook 圖表：更新次數與 epoch |
| 36、37、38、51 | 04-17 | 表 4-1：資料規模與可用訓練方式；表 4-1：超參數、縮放與實作；fit、partial_fit 與 epoch 的差別；Notebook 圖表：更新次數與 epoch |
| 39、40、42、43、44、46 | 04-18 | 資料是彎曲的，直線可能系統性失準；多項式迴歸：先轉換特徵，再用線性模型；模型階數：太簡單與太靈活的兩端；學習曲線：訓練與驗證誤差都偏高；學習曲線：訓練很好，驗證落後；診斷工作單：下一步該改什麼？ |
| 39、40、41、42 | 04-19 | 資料是彎曲的，直線可能系統性失準；多項式迴歸：先轉換特徵，再用線性模型；多項式特徵也包含交互作用；模型階數：太簡單與太靈活的兩端 |
| 45、52 | 04-20 | 偏差與變異：重複抽樣後會看到什麼？；Notebook 圖表：重複抽樣的偏差與變異 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 04-01｜投影片 04：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 04-02｜投影片 04：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. 三種最小平方解法：正規方程、lstsq 與 SVD 偽反矩陣

資料生成規則 $y = 4 + 3x + \varepsilon$。三種解法在滿欄秩時給相同係數；程式一般用 `lstsq`／SVD，不自行反矩陣。

In [ ]:
# 程式 04-03｜投影片 04：5、7、8、9、15、16、17、18 頁
rng = np.random.default_rng(SEED)
x = rng.uniform(0, 2, size=(120, 1))
y = 4 + 3 * x[:, 0] + rng.normal(0, 1, size=120)
Xb = np.c_[np.ones(len(x)), x]                      # 第一欄為截距

theta_normal = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y # 正規方程：需要 X^T X 可逆
theta_lstsq = np.linalg.lstsq(Xb, y, rcond=None)[0] # 最小平方求解器
theta_pinv = np.linalg.pinv(Xb) @ y                # SVD 偽反矩陣 X^+ y

from sklearn.linear_model import LinearRegression
sk = LinearRegression().fit(x, y)
assert np.allclose(theta_normal, theta_lstsq) and np.allclose(theta_lstsq, theta_pinv)
assert np.allclose(theta_lstsq, [sk.intercept_, sk.coef_[0]])
pd.DataFrame({'normal_equation': theta_normal, 'lstsq': theta_lstsq, 'pinv_SVD': theta_pinv},
             index=['intercept', 'slope'])

## 直線資料與最小平方配適

對應「帶有雜訊的直線資料」與「估計直線與資料生成直線」。採固定生成規則與獨立亂數樣本，重點是比較資料生成直線與估計直線。

In [ ]:
# 程式 04-04｜投影片 04：9、16 頁
x_grid = np.linspace(0, 2, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax in axes:
    ax.scatter(x[:, 0], y, s=15, alpha=.5)
    ax.set(xlabel='x', ylabel='y')
axes[0].set_title('Noisy observations')
axes[1].plot(x_grid, 4+3*x_grid, 'k--', label='Generating mean')
axes[1].plot(x_grid, theta_lstsq[0]+theta_lstsq[1]*x_grid, label='Least squares')
axes[1].legend(); savefig('c04_linear_fit')

In [ ]:
# 程式 04-05｜投影片 04：17、50 頁
# 共線時以秩判斷不可識別性；浮點 inv 不保證一定拋出例外。
Xc = np.c_[np.ones(len(x)), x[:, 0], 2 * x[:, 0]]
rank_c = np.linalg.matrix_rank(Xc)
theta_c = np.linalg.lstsq(Xc, y, rcond=None)[0]
assert rank_c < Xc.shape[1]
assert np.isclose(theta_c[1] + 2 * theta_c[2], theta_lstsq[1], atol=1e-6)
assert np.allclose(Xc @ theta_c, Xb @ theta_lstsq)
print('rank / columns:', rank_c, Xc.shape[1])
print('min-norm solution:', theta_c, 'condition number:', np.linalg.cond(Xc.T @ Xc))

## 2. 手算驗算：三個點決定哪一條線？

對應投影片活動「三個點能決定哪一條線？」。資料 $(0,1),(1,3),(2,5)$，含截距的 $X$ 三列為 $[1,0],[1,1],[1,2]$。

In [ ]:
# 程式 04-06｜投影片 04：8、20 頁
X3 = np.array([[1., 0.], [1., 1.], [1., 2.]])
y3 = np.array([1., 3., 5.])
def mse(theta, X, y):
    return float(np.mean((X @ theta - y) ** 2))
row = {'theta=(1,2)': mse(np.array([1., 2.]), X3, y3),
       'theta=(0,2)': mse(np.array([0., 2.]), X3, y3)}
assert row['theta=(1,2)'] == 0.0 and row['theta=(0,2)'] == 1.0
pd.Series(row, name='MSE')

## 置中公式與微分驗算

定義 $r=\hat y-y$、$J=\mathrm{MSE}$。置中公式給 $b=1,w=2$；有限差分 $[J(\theta+h e_j)-J(\theta-h e_j)]/(2h)$ 用來檢查解析梯度，不能取代符號推導。

In [ ]:
# 程式 04-07｜投影片 04：10、11、12、48、50 頁
x3 = X3[:, 1]
w_centered = ((x3-x3.mean())*(y3-y3.mean())).sum()/((x3-x3.mean())**2).sum()
b_centered = y3.mean()-w_centered*x3.mean()
assert np.allclose([b_centered,w_centered], [1,2])
assert np.isclose(((x3-x3.mean())*(y3-y3.mean())).sum(), (x3*y3).sum()-len(x3)*x3.mean()*y3.mean())
def gradient(theta, X, y):
    return 2/len(X)*X.T@(X@theta-y)
probe_theta = np.array([.3, .8]); h = 1e-5
numerical = np.array([(mse(probe_theta+h*e,X3,y3)-mse(probe_theta-h*e,X3,y3))/(2*h) for e in np.eye(2)])
analytic = gradient(probe_theta,X3,y3)
assert np.allclose(numerical,analytic,atol=1e-7)
pd.DataFrame({'analytic':analytic,'finite_difference':numerical},index=['b','w'])

## 多元迴歸：每個特徵的梯度

對應多元矩陣公式及工作單 E。$X$ 含截距欄，$\theta=(b,w_1,w_2)^\top$；同時用逐欄求和與矩陣乘法算梯度。

In [ ]:
# 程式 04-08｜投影片 04：7、13、14 頁
Xm = np.array([[1.,1.,0.],[1.,0.,1.],[1.,1.,1.]])
ym = np.array([1.,2.,3.]); theta_m = np.zeros(3)
gm = gradient(theta_m,Xm,ym)
component = np.array([2*np.mean((Xm@theta_m-ym)*Xm[:,j]) for j in range(3)])
assert np.allclose(gm,component) and np.allclose(gm,[-4,-8/3,-10/3])
updated_m = theta_m-.15*gm
assert np.allclose(updated_m,[.6,.4,.5]) and np.isclose(mse(updated_m,Xm,ym),1.02)
print('X, theta, prediction shapes:',Xm.shape,theta_m.shape,(Xm@theta_m).shape)
pd.Series({'gradient':gm.tolist(),'updated':updated_m.tolist(),'MSE_after':mse(updated_m,Xm,ym)})

## 3. 批次梯度下降：學習率決定收斂、偏慢或發散

在標準化座標下，MSE 梯度為 $\nabla_\theta J = \frac{2}{m}X^\top(X\theta-y)$。比較三個學習率下的收斂情形。

In [ ]:
# 程式 04-09｜投影片 04：3、22、23、24、25 頁
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(x)
Z = np.c_[np.ones(len(x)), scaler.transform(x)]

def batch_gd(Z, y, eta, epochs=60):
    theta = np.zeros(Z.shape[1]); hist = []
    for _ in range(epochs):
        err = Z @ theta - y
        hist.append(np.mean(err ** 2))
        theta = theta - eta * (2 / len(Z)) * Z.T @ err
    return theta, np.array(hist)

rows = []
for eta, label in [(0.02, 'too small'), (0.3, 'good'), (1.05, 'diverges')]:
    theta, hist = batch_gd(Z, y, eta)
    rows.append({'eta': eta, 'note': label, 'final_logged_MSE': hist[-1]})
    plt.semilogy(np.clip(hist, 1e-3, 1e12), label=f'eta={eta} ({label})')
plt.xlabel('Iteration'); plt.ylabel('Training MSE (log scale)'); plt.legend()
savefig('09_gradient_learning_rate')

theta_good, _ = batch_gd(Z, y, 0.3, epochs=400)
theta_closed = np.linalg.lstsq(Z, y, rcond=None)[0]
assert np.allclose(theta_good, theta_closed, atol=1e-4)
assert rows[2]['final_logged_MSE'] > rows[0]['final_logged_MSE']       # eta=1.05 發散
pd.DataFrame(rows)

## 單參數損失、步幅與非凸反例

教材圖 4-3～4-6 的可計算示意。前三圖用 $f(u)=(u-3)^2$；另畫 $f(u)=(u^2-1)^2$ 說明非凸情形。非凸圖不是線性迴歸的 MSE。

In [ ]:
# 程式 04-10｜投影片 04：22、24、25、26 頁
fig, axes = plt.subplots(1,3,figsize=(12,3.5))
u_grid = np.linspace(-1,7,400)
for ax, eta in zip(axes,[.05,.3,1.05]):
    u = 0.; trajectory = [u]
    for _ in range(6):
        u -= eta*2*(u-3); trajectory.append(u)
    trajectory = np.array(trajectory)
    ax.plot(u_grid,(u_grid-3)**2)
    ax.plot(trajectory,(trajectory-3)**2,'o-',label=f'eta={eta}')
    ax.set(xlabel='parameter u',ylabel='loss');ax.legend()
savefig('c04_descent_steps')
u_grid = np.linspace(-1.6,1.6,400)
plt.figure();plt.plot(u_grid,(u_grid**2-1)**2,label='Nonconvex: two minima')
plt.scatter([-1,0,1],[0,1,0]);plt.xlabel('u');plt.ylabel('loss');plt.legend()
savefig('c04_nonconvex')

## 標準化前後的損失等高線

使用兩個尺度相差 20 倍的特徵，固定截距，只觀察兩個權重。圖中的權重座標因縮放而不同；比較等高線形狀及更新路徑，不直接比較權重數值。每種尺度依自己的最大曲率選穩定步幅。

In [ ]:
# 程式 04-11｜投影片 04：27 頁
rng_scale = np.random.default_rng(SEED)
features = rng_scale.normal(size=(200,2))*[1,20]
features -= features.mean(axis=0)
y_scale = features@np.array([1.,.1])
scaled_features = StandardScaler().fit_transform(features)
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, features_here, label in zip(axes,[features,scaled_features],['Original scales','Standardized']):
    optimum = np.linalg.lstsq(features_here,y_scale,rcond=None)[0]
    a = np.linspace(-.5,max(2,optimum[0]*1.3),120)
    b = np.linspace(-.5,max(2,optimum[1]*1.3),120)
    aa,bb = np.meshgrid(a,b)
    candidates=np.c_[aa.ravel(),bb.ravel()]
    surface=np.mean((features_here@candidates.T-y_scale[:,None])**2,axis=0).reshape(aa.shape)
    ax.contour(aa,bb,surface,levels=np.geomspace(.02,100,12))
    rate=.9/np.linalg.eigvalsh(2*features_here.T@features_here/len(features_here)).max()
    w=np.zeros(2);path=[w.copy()]
    for _ in range(40):
        w-=rate*gradient(w,features_here,y_scale);path.append(w.copy())
    path=np.array(path);ax.plot(path[:,0],path[:,1],'r.-');ax.plot(*optimum,'k*')
    ax.set(title=label,xlabel='weight 1',ylabel='weight 2');ax.set_aspect('equal',adjustable='box')
savefig('c04_scaling_contours')

## 不同學習率的前幾條預測線

資料、起點與更新次數固定，只改學習率。計算在標準化特徵上，圖仍以原始 x 為橫軸。

In [ ]:
# 程式 04-12｜投影片 04：30 頁
grid_Z=np.c_[np.ones(len(x_grid)),scaler.transform(x_grid[:,None])]
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,eta in zip(axes,[.02,.3,1.05]):
    theta_draw=np.zeros(2);ax.scatter(x[:,0],y,s=8,alpha=.25)
    for step in range(6):
        ax.plot(x_grid,grid_Z@theta_draw,alpha=.7,label=str(step))
        theta_draw-=eta*gradient(theta_draw,Z,y)
    ax.set(title=f'eta={eta}',xlabel='x',ylabel='prediction')
axes[-1].legend(title='step',fontsize=8);savefig('c04_line_updates')

## 4. 手算驗算：一次更新後損失是否下降？

對應投影片活動「手算一次更新」。$X=\begin{bmatrix}1&0\\1&1\end{bmatrix}$、$y=(1,3)^\top$、$\theta=(0,0)^\top$、$\eta=0.1$。

In [ ]:
# 程式 04-13｜投影片 04：23、28、29 頁
Xa = np.array([[1., 0.], [1., 1.]]); ya = np.array([1., 3.]); theta = np.zeros(2)
resid = Xa @ theta - ya                       # [-1, -3]
grad = (2 / len(Xa)) * Xa.T @ resid           # [-4, -3]
theta_new = theta - 0.1 * grad                # [0.4, 0.3]
assert np.allclose(resid, [-1, -3]) and np.allclose(grad, [-4, -3])
assert np.allclose(theta_new, [0.4, 0.3])
assert np.isclose(mse(theta, Xa, ya), 5.0) and np.isclose(mse(theta_new, Xa, ya), 2.825)
pd.Series({'residual': resid.tolist(), 'gradient': grad.tolist(), 'theta_new': theta_new.tolist(),
           'MSE_before': mse(theta, Xa, ya), 'MSE_after': mse(theta_new, Xa, ya)})

## 負梯度的一階近似與同步更新

比較精確的新 MSE 與一階近似。平方損失的餘項是 $\eta^2\|Xg\|^2/m$；因此過大步幅可能使損失增加。

In [ ]:
# 程式 04-14｜投影片 04：23、49 頁
eta_rows=[]
for rate in [.001,.01,.1,1.]:
    exact=mse(theta-rate*grad,Xa,ya)
    first=mse(theta,Xa,ya)-rate*(grad@grad)
    remainder=rate**2*np.mean((Xa@grad)**2)
    assert np.isclose(exact,first+remainder)
    eta_rows.append({'eta':rate,'exact_MSE':exact,'first_order':first,'remainder':remainder})
wrong=theta.copy();wrong[0]-=.1*grad[0]
wrong[1]-=.1*gradient(wrong,Xa,ya)[1]
assert np.allclose(wrong,[.4,.26])
print('Incorrect sequential update:',wrong,'correct simultaneous:',theta_new)
pd.DataFrame(eta_rows)

## 5. SGD、mini-batch 與學習率排程

`SGDRegressor` 逐筆更新、每個 epoch 打亂資料；學習率可用排程 $\eta_t = t_0/(t+t_1)$ 逐步縮小。下圖比較各 epoch 結束時的參數位置；每個 epoch 的更新次數不同，不是相同運算成本的競賽。手寫 SGD 採此排程，下面 SGDRegressor 使用套件預設排程，兩者不要求逐步相同。

In [ ]:
# 程式 04-15｜投影片 04：31、32、34、36、37 頁
from sklearn.linear_model import SGDRegressor
target = y

def run_path(kind, epochs=25, batch=1, eta=0.1, every_update=False):
    theta = np.zeros(Z.shape[1]); path = [theta.copy()]
    idx = np.arange(len(Z)); step = 0
    for _ in range(epochs):
        np.random.default_rng(SEED + _).shuffle(idx)
        if kind == 'batch':
            g = (2 / len(Z)) * Z.T @ (Z @ theta - target)
            theta = theta - eta * g; path.append(theta.copy()); continue
        for s in range(0, len(Z), batch):
            step += 1
            b = idx[s:s + batch]
            g = (2 / len(b)) * Z[b].T @ (Z[b] @ theta - target[b])
            lr = eta if kind == 'minibatch' else 5.0 / (step + 50)   # SGD 用排程
            theta = theta - lr * g
            if every_update: path.append(theta.copy())
        if not every_update: path.append(theta.copy())
    return np.array(path)

for kind, kw in [('batch', {}), ('minibatch', {'batch': 16}), ('sgd', {'batch': 1})]:
    p = run_path(kind, **kw)
    plt.plot(p[:, 0], p[:, 1], marker='.', alpha=.8, label=kind)
plt.scatter(theta_closed[0], theta_closed[1], marker='*', s=180, c='k', label='least squares')
plt.title('Epoch-end checkpoints; updates per epoch differ')
plt.xlabel(r'$\theta_0$ intercept (standardized)'); plt.ylabel(r'$\theta_1$ slope'); plt.legend()
savefig('09_gd_paths')

sgd = SGDRegressor(penalty=None, max_iter=1000, tol=1e-4, random_state=SEED).fit(Z[:, 1:], y)
print('SGDRegressor coef vs least squares (standardized):',
      np.round(sgd.coef_, 3), np.round(theta_closed[1:], 3))

## SGD 每次更新與前幾條預測線

對應圖 4-9、4-10、4-11。第一圖記錄每次更新的整體 MSE，第二圖畫 SGD 前六次更新的直線；單筆梯度可能使整體損失暫時增加。

In [ ]:
# 程式 04-16｜投影片 04：31、33、34、51 頁
fig,axes=plt.subplots(1,2,figsize=(11,4))
for kind,kw in [('batch',{}),('minibatch',{'batch':16}),('sgd',{'batch':1})]:
    points=run_path(kind,epochs=8,every_update=True,**kw)
    losses=np.array([mse(t,Z,y) for t in points])
    axes[0].semilogy(np.arange(len(points)),losses,label=kind)
    axes[1].plot(points[:,0],points[:,1],'.-',alpha=.6,label=kind)
axes[0].set(xlabel='Parameter updates (different cost per update)',ylabel='Full training MSE')
axes[1].set(xlabel='Intercept (standardized)',ylabel='Slope (standardized)')
for ax in axes: ax.legend()
savefig('c04_stochastic_updates')
sgd_first=run_path('sgd',epochs=1,every_update=True)[:6]
plt.figure();plt.scatter(x[:,0],y,s=12,alpha=.4)
for k,parameters in enumerate(sgd_first): plt.plot(x_grid,grid_Z@parameters,label=f'step {k}')
plt.xlabel('x');plt.ylabel('prediction');plt.legend();savefig('c04_sgd_lines')

## fit、partial_fit 與 epoch

一個 epoch 是看過一輪資料；本例每輪分成 4 塊，呼叫 4 次 partial_fit。SGDRegressor 的 squared_error 使用半平方損失，與本章手寫 MSE 的梯度尺度不同，不逐步比較相同 eta。

In [ ]:
# 程式 04-17｜投影片 04：36、37、38、51 頁
stream=SGDRegressor(penalty=None,learning_rate='constant',eta0=.01,random_state=SEED)
stream_rows=[]
for epoch in range(20):
    for block in np.array_split(np.random.default_rng(SEED+epoch).permutation(len(Z)),4):
        stream.partial_fit(Z[block,1:],y[block])
    stream_rows.append({'epoch':epoch+1,'calls':4*(epoch+1),'MSE':np.mean((stream.predict(Z[:,1:])-y)**2)})
assert np.isfinite(stream.coef_).all()
pd.DataFrame(stream_rows).iloc[[0,4,9,19]]

## 6. 多項式迴歸與學習曲線

多項式先轉換特徵再套線性模型，對參數仍是線性。學習曲線把訓練與驗證誤差畫成資料量的函數，用來分辨欠擬合（兩線都高且接近）與過擬合（兩線有明顯落差）。

本檔學習曲線的縱軸採對數尺度，以清楚呈現小樣本時較大的驗證誤差。

In [ ]:
# 程式 04-18｜投影片 04：39、40、42、43、44、46 頁
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import learning_curve
from sklearn.metrics import root_mean_squared_error

rng = np.random.default_rng(SEED)
xc = rng.uniform(-3, 3, size=(200, 1))
yc = 0.5 * xc[:, 0] ** 2 + xc[:, 0] + 2 + rng.normal(0, 1, 200)   # 彎曲資料

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sizes = np.linspace(0.15, 1.0, 8)
for degree, ax in [(1, axes[0]), (10, axes[1])]:
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
    ts, tr, va = learning_curve(model, xc, yc, train_sizes=sizes, cv=5,
                                scoring='neg_root_mean_squared_error')
    ax.plot(ts, -tr.mean(1), 'o-', label='train RMSE')
    ax.plot(ts, -va.mean(1), 's-', label='CV RMSE')
    ax.set(title=f'degree {degree}', xlabel='training samples', ylabel='RMSE (log scale)', yscale='log'); ax.legend()
savefig('09_learning_curves')

gap = {}
for degree in [1, 2, 10]:
    m = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression()).fit(xc, yc)
    ts, tr, va = learning_curve(m, xc, yc, train_sizes=sizes, cv=5, scoring='neg_root_mean_squared_error')
    gap[degree] = {'train_RMSE_full': float(-tr.mean(1)[-1]), 'CV_RMSE_full': float(-va.mean(1)[-1])}
assert gap[1]['train_RMSE_full'] > gap[2]['train_RMSE_full']            # degree 1 欠擬合
assert gap[10]['CV_RMSE_full'] - gap[10]['train_RMSE_full'] > gap[2]['CV_RMSE_full'] - gap[2]['train_RMSE_full']
report('linear_optimization', {'least_squares_theta': theta_closed.tolist(),
       'gd_learning_rates': rows, 'learning_curve_gap': gap})
pd.DataFrame(gap).T

## 彎曲資料、二次配適與模型階數

對應圖 4-12～4-14。固定一份資料，比較 1、2、10 次模型；圖形是觀察假設，泛化仍由前一節的 CV 學習曲線判斷。

In [ ]:
# 程式 04-19｜投影片 04：39、40、41、42 頁
grid_curve=np.linspace(-3,3,300)[:,None]
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax in axes: ax.scatter(xc[:,0],yc,s=8,alpha=.3);ax.set(xlabel='x',ylabel='y')
axes[0].set_title('Curved observations')
for degree in [1,2,10]:
    polynomial=make_pipeline(PolynomialFeatures(degree,include_bias=False),StandardScaler(),LinearRegression()).fit(xc,yc)
    prediction_curve=polynomial.predict(grid_curve)
    axes[2].plot(grid_curve[:,0],prediction_curve,label=f'degree {degree}')
    if degree==2: axes[1].plot(grid_curve[:,0],prediction_curve,label='degree 2')
axes[1].legend();axes[2].legend();savefig('c04_polynomial_fits')
poly2=PolynomialFeatures(2,include_bias=False)
expanded=poly2.fit_transform([[2.,3.],[4.,5.]])
assert np.allclose(expanded[0],[2,3,4,6,9])
pd.DataFrame(expanded,columns=poly2.get_feature_names_out(['x1','x2']))

## 重複抽樣下的偏差與變異

對應圖 4-17 的概念。固定 $f(x)=0.5x^2+x+2$、雜訊變異數 1，重抽 100 組訓練集。圖中是固定 x 下跨訓練集的平均預測與標準差；不是參數信賴區間。使用已知生成函數，才可估計偏差平方。

In [ ]:
# 程式 04-20｜投影片 04：45、52 頁
rng_repeat=np.random.default_rng(SEED)
grid_repeat=np.linspace(-2.8,2.8,100)[:,None]
truth=.5*grid_repeat[:,0]**2+grid_repeat[:,0]+2
predictions={d:[] for d in [1,2,10]}
for _ in range(100):
    xr=rng_repeat.uniform(-3,3,(60,1));yr=.5*xr[:,0]**2+xr[:,0]+2+rng_repeat.normal(0,1,60)
    for degree in predictions:
        reg=make_pipeline(PolynomialFeatures(degree,include_bias=False),StandardScaler(),LinearRegression()).fit(xr,yr)
        predictions[degree].append(reg.predict(grid_repeat))
fig,axes=plt.subplots(1,3,figsize=(12,3.5));bv_rows=[]
for ax,(degree,values) in zip(axes,predictions.items()):
    values=np.array(values);mean=values.mean(axis=0);variance=values.var(axis=0)
    bias2=(mean-truth)**2
    assert np.allclose(np.mean((values-truth)**2,axis=0),bias2+variance)
    ax.plot(grid_repeat[:,0],truth,'k--',label='truth');ax.plot(grid_repeat[:,0],mean,label='mean prediction')
    ax.fill_between(grid_repeat[:,0],mean-np.sqrt(variance),mean+np.sqrt(variance),alpha=.25,label='+/- 1 SD')
    ax.set(title=f'degree {degree}',xlabel='x',ylabel='prediction');ax.legend(fontsize=8)
    bv_rows.append({'degree':degree,'bias_squared':bias2.mean(),'variance':variance.mean(),'expected_MSE_estimate':bias2.mean()+variance.mean()+1})
savefig('c04_bias_variance')
pd.DataFrame(bv_rows)

## 練習與參考答案

- 共線時為什麼不能直接反轉 $X^\top X$？**矩陣奇異（欄不獨立），反矩陣不存在；`lstsq`／SVD 給出最小範數解，但個別係數仍不可辨識。**
- 學習率調小能讓直線變成拋物線嗎？**不能，那是模型表達能力問題；要改特徵或模型。**
- 訓練與驗證誤差相近且都很高時該怎麼辦？**先查欠擬合：換更合適的特徵或模型，不是只蒐集更多資料。**